# FCA–MoE v0.3 — Qwen1.5-MoE-A2.7B real routing extraction

**Model:** `Qwen/Qwen1.5-MoE-A2.7B` · **24 MoE layers** · **60 routed experts** · **Top-4** · **4 logical shared experts**.

This notebook captures actual router outputs from each `Qwen2MoeTopKRouter` and separately captures the scalar sigmoid gate controlling the merged shared-expert MLP. The Hugging Face implementation exposes one merged shared MLP, not four independently hookable shared-expert modules.

## T4 precision and memory constraints

The checkpoint declares BF16. A Tesla T4 is not a native BF16 compute target, so this pilot explicitly loads/converts weights to FP16. The resulting traces describe the FP16 runtime, not bit-identical BF16 routing. The checkpoint has about 14.3B total parameters: 2.7B active parameters per token does **not** mean only 2.7B parameters need to be stored. GPU/CPU/disk offload is required and model loading/forward execution are empirical gates.

The initial pilot is four 256-token WikiText-2 test sequences. No successful load or trace collection is assumed.

In [ ]:
# Keep Colab's existing PyTorch/CUDA build. Remove incompatible optional audio/vision extensions
# because this is a text-only model and they are not required.
%pip -q install -U "transformers==5.22.2" accelerate datasets pyarrow pandas psutil
%pip -q uninstall -y torchvision torchaudio
print("Installation cell complete. Restart the runtime before continuing.")

**Restart runtime now:** Runtime → Restart session. Then continue with the next cell. This avoids stale imported binary extensions.

In [ ]:
import torch, transformers, psutil, platform, importlib.util
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
print("Available host RAM GiB:", round(psutil.virtual_memory().available / 2**30, 2))
print("torchvision:", importlib.util.find_spec("torchvision"))
print("torchaudio:", importlib.util.find_spec("torchaudio"))
if not torch.cuda.is_available(): raise RuntimeError("CUDA GPU not detected")
cap = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), "SM:", cap, "free/total GiB:", tuple(round(x/2**30,2) for x in torch.cuda.mem_get_info(0)))
if cap < (7,5): raise RuntimeError("This pilot expects compute capability 7.5 or newer")
from transformers import AutoConfig, AutoModelForCausalLM
print("Transformers model import: OK")

In [ ]:
from transformers import AutoConfig
MODEL_ID = "Qwen/Qwen1.5-MoE-A2.7B"
config = AutoConfig.from_pretrained(MODEL_ID)
print("Architecture:", config.architectures)
print("Model type:", config.model_type)
print("Layers:", config.num_hidden_layers)
print("Routed experts:", config.num_experts)
print("Top-K:", config.num_experts_per_tok)
print("norm_topk_prob:", config.norm_topk_prob)
print("Shared expert intermediate size:", config.shared_expert_intermediate_size)
assert config.model_type == "qwen2_moe"
assert config.num_hidden_layers == 24 and config.num_experts == 60 and config.num_experts_per_tok == 4
assert config.norm_topk_prob is False
print("Architecture preflight: PASSED")

## Mount Drive and obtain the research branch

Parquet shards and the manifest persist on Drive. Model offload files use the Colab VM local disk.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib
REPO_DIR = pathlib.Path('/content/FCA-MoE')
if not REPO_DIR.exists():
    !git clone --depth 1 --branch research/fca-moe-v0.3 https://github.com/msellamiTN/FCA-MoE.git /content/FCA-MoE
%cd /content/FCA-MoE/research/v03
!git pull --ff-only origin research/fca-moe-v0.3

In [ ]:
# Four-sequence pilot. FP16 conversion is explicit and recorded in the manifest.
OUTPUT_DIR = "/content/drive/MyDrive/FCA_MoE_v03/qwen15_moe_a27_wikitext_test"
!python scripts/extract_qwen15_moe_routes.py \
  --output-dir "$OUTPUT_DIR" \
  --sequence-length 256 \
  --max-chunks 4 \
  --gpu-memory 12GiB \
  --cpu-memory 6GiB \
  --min-cpu-ram-gib 6

In [ ]:
# Validate captured router routes and shared-gate values.
!python scripts/validate_qwen15_moe_traces.py --output-dir "$OUTPUT_DIR"

## Resume and scientific interpretation

After interruption, rerun the extraction command with the same output directory and configuration. It resumes from the next contiguous committed Parquet shard. Do not change model revision, dataset revision, sequence length, or runtime dtype within an existing checkpoint directory.

The trace phase is `prefill_teacher_forced`, not autoregressive decode. `score_0..score_3` are raw selected softmax probabilities because `norm_topk_prob=False`; they generally do **not** sum to one. `shared_gate_weight` is the sigmoid scalar applied to the merged shared-expert MLP output. These traces do not by themselves establish KV-cache reuse or inference speedup.